# 17)
## (i)
Let $X,Y \sim Unif(0,1)$ be iid, then let $Z = \mathbf{1}(X^2 +Y^2 \leq 1)$. Note that this defines $Z \sim Bernoulli(p).$ We are interested in the expectation of $Z$, which is given by
$$
E[Z] = P(Z=1) = P(X^2 + Y^2 \leq 1) = \frac{\pi}{4}
$$
However, we wish to estimate this expectation using conditional Monte Carlo. We can condition on $X$, and note that
$$
E[Z|X] = 0\cdot P(Z = 0| X = x) + 1\cdot P(Z=1|X = x) = P(Y^2 \leq 1 - x^2) =
$$
$$
P(Y \leq \sqrt{1-x^2}) = \sqrt{1-x^2}
$$
Since Y is uniform on (0,1), we have that $P(Y \leq y) = y$ for $y \in (0,1)$. Now since we simulate $\ell_c = N^{-1}\sum_{i=1}^{N} E[Z|X_i]$ where $X_i \sim Unif(0,1)$, we have that $\ell_c \xrightarrow{p} E[E[Z|X]] = E[Z]$ is an unbiased estimator of $E[Z]$. We can then use the law of total variance to show that the variance of $\ell_c$ is less than the variance of the naive estimator $\ell$. Given $E[Z] = \pi/4$, and since $Z$ is Bernoulli, we have that $Z^2 = Z$, and thus
$$Var[Z] = \dfrac{\pi}{4} - \dfrac{\pi^2}{16}.$$

However, by the law of total variation, we have that
$$
Var(E[Z|X]) = Var(Z) - E[Var(Z|X)] \leq Var(Z).
$$
Now $Z|X = x \sim Bernoulli(P(Z=1| X = x)) = Bernoulli(\sqrt{1-x^2})$, and thus $Var(Z|X=x) = \sqrt{1-x^2}(1-\sqrt{1-x^2})$. We can then compute the expectation of this variance as
$$
E[Var(Z|X)] = \int_{0}^{1} \sqrt{1-x^2}(1-\sqrt{1-x^2}) dx = \int_{0}^{1} \sqrt{1-x^2} dx - \int_{0}^{1} (1-x^2) dx = \frac{\pi}{4} - \frac{2}{3}
$$
So conditional Monte Carlo gives a variance reduction of $\frac{\pi}{4} - \frac{2}{3} \approx 0.12 $

In [1]:
import numpy as np
N = 1000 # Samples
M = 1000 # Simulations
# Crude montecarlo
var = np.pi/4*(1-np.pi/4)
U = np.random.uniform(0,1,(N,M))
V =np.random.uniform(0,1, (N,M))
Z = U**2 + V**2 <= 1
L = np.mean(Z, axis=0)
print("Crude")
print("Variation of crude montecarlo", np.var(L))
print("Theoretical variance:",  var / N)
print()
# Conditional montecarlo, closed form of conditional expectation
print("Conditional")
L_c = np.mean(np.sqrt(1-U**2), axis=0)
print("Variation of conditional montecarlo", np.var(L_c))
print("Theoretical variance:", (var - (np.pi / 4 - 2/3)) / N)

print("Improvement" , np.var(L)/np.var(L_c))

Crude
Variation of crude montecarlo 0.00016799219900000018
Theoretical variance: 0.0001685478883293634

Conditional
Variation of conditional montecarlo 5.052548387237179e-05
Theoretical variance: 4.981639159858175e-05
Improvement 3.32490034977896


## (ii)
We can also use conditional Monte Carlo on task 5.

We can estimate the integral $\int_0^\pi sin(x)dx$ by letting $U,V \sim Unif(0,\pi)$, iid. Then we can define $Z = \mathbf{1}(V \leq sin(U))$. If we then condition on $U$, we have that
$$
E[Z|U = u] = 0\cdot P(Z=0|U) + 1\cdot P(Z=1|U) = P(V \leq sin(U)) = \frac{sin(U)}{\pi}
$$
Now we see that by LIE,
$$E[Z] = E[E[Z|U]] = E[sin(U)/\pi] = \frac{1}{\pi^2}\int_0^\pi sin(x)dx$$

We know that $\int_0^\pi sin(x)dx = 2$, and that $Z = Z^2$, so the variance of the crude Monte Carlo estimator is given by
$$
Var[Z] = E[Z^2] - E[Z]^2 = E[Z] - E[Z]^2 = \frac{2}{\pi^2} - \frac{4}{\pi^4}
$$
Now we can compute the variance of the conditional Monte Carlo estimator. We have that $Z|U = u \sim Bernoulli(P(Z=1|U=u)) = Bernoulli(sin(u)/\pi)$, and thus

$$
\begin{aligned}
\operatorname{Var}(E[Z\mid U])
&= E\left[(E[Z\mid U])^2\right]
   - \left(E[E[Z\mid U]]\right)^2 \\[4pt]
&= E\left[\left(\frac{\sin(U)}{\pi}\right)^2\right]
   - \left(\frac{2}{\pi^2}\right)^2 \\[4pt]
&= \frac{1}{\pi}\int_0^\pi
   \left(\frac{\sin(x)}{\pi}\right)^2\,dx
   - \frac{4}{\pi^4} \\[4pt]
&= \frac{1}{\pi^3}\int_0^\pi \sin^2(x)\,dx
   - \frac{4}{\pi^4}.
\end{aligned}
$$

Using

$$
\sin^2(x)=\frac{1-\cos(2x)}{2},
$$

we get

$$
\begin{aligned}
\int_0^\pi \sin^2(x)\,dx
&= \frac{1}{2}\int_0^\pi (1-\cos(2x))\,dx \\[4pt]
&= \frac{1}{2}
\left[x-\frac{\sin(2x)}{2}\right]_0^\pi \\[4pt]
&= \frac{\pi}{2}.
\end{aligned}
$$

Therefore,

$$
\boxed{
\operatorname{Var}(E[Z\mid U])
=
\frac{1}{2\pi^2}
-\frac{4}{\pi^4}
}
$$

In [3]:
N = 1000 # Samples
M = 1000 # Simulations
# Crude montecarlo
var = 2/np.pi**2 - 4/np.pi**4
U = np.random.uniform(0,np.pi,(N,M))
V =np.random.uniform(0,np.pi, (N,M))
Z = V <= np.sin(U)
L = np.mean(Z, axis=0)
print("Crude")
print("Variation of crude montecarlo", np.var(L))
print("Theoretical variance:",  var / N)
print()
# Conditional montecarlo, closed form of conditional expectation
print("Conditional")
L_c = np.mean(np.sin(U) / np.pi, axis=0)
print("Variation of conditional montecarlo", np.var(L_c))
print("Theoretical variance:", (1/(2*np.pi**2) - 4/np.pi**4) / N)

print("Improvement" , np.var(L)/np.var(L_c))

Crude
Variation of crude montecarlo 0.00017640353599999997
Theoretical variance: 0.00016157843826593822

Conditional
Variation of conditional montecarlo 9.778236202030488e-06
Theoretical variance: 9.596662802431537e-06
Improvement 18.04042491460465
